 # **1. Import Dependencies and Page Setup**

 This first cell creates the `app.py` file from scratch (overwriting any old version).

In [9]:
# %%
import ee
# Run this once in the notebook to generate the credentials file
ee.Authenticate()
ee.Initialize(project='davi-carbono-ufjf')

In [10]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [11]:
%%writefile app.py

# ==========================================
# IMPORT DEPENDENCIES
# ==========================================
import os
import pandas as pd
import numpy as np
import ee
import folium
from streamlit_folium import folium_static
import joblib
import streamlit as st



Overwriting app.py


 # **2. Page Configuration & Authentication**

 Notice the `-a` tag below. It appends this block to the `app.py` file.

In [12]:

%%writefile -a app.py

# ==========================================
# PAGE CONFIGURATION & AUTHENTICATION
# ==========================================
# Set up the basic layout and title for the web interface
st.set_page_config(page_title="Carbon Prediction Dashboard", layout="wide")

# Read the credentials automatically generated by the Notebook cell
try:
    ee.Initialize(project='davi-carbono-ufjf')
except Exception as e:
    st.error("Earth Engine initialization failed. Please authenticate in the notebook first.")

Appending to app.py


 # **3. MapBiomas Spatial Assets & Color Palettes**

In [13]:
%%writefile -a app.py

# ==========================================
# MAPBIOMAS ASSETS & COLOR PALETTES
# ==========================================
# Define correct GPP product path from MapBiomas Collection 9
path_project = 'projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_pasture_gpp_v1'
mapbiomas = ee.Image(path_project)

# Hexadecimal color codes dictionary
dicionario_cores = {
    1: "#32a65e", 3: "#1f8d49", 4: "#7dc975", 5: "#04381d", 6: "#026975",
    49: "#02d659", 10: "#ad975a", 11: "#519799", 12: "#d6bc74", 32: "#fc8114",
    29: "#ffaa5f", 50: "#ad5100", 13: "#d89f5c", 14: "#FFFFB2", 15: "#edde8e",
    18: "#E974ED", 19: "#C27BA0", 39: "#f5b3c8", 20: "#db7093", 40: "#c71585",
    62: "#ff69b4", 41: "#f54ca9", 36: "#d082de", 46: "#d68fe2", 47: "#9932cc",
    48: "#e6ccff", 9: "#7a5900", 21: "#ffefc3", 22: "#d4271e", 23: "#ffa07a",
    24: "#d4271e", 30: "#9c0027", 25: "#db4d4f", 26: "#0000FF", 33: "#2532e4",
    31: "#091077", 27: "#ffffff"
}

# Land use and land cover classes corresponding to the color codes mapping
dicionario_classes = {
    1: "Floresta", 3: "Formação Florestal", 4: "Formação Savânica", 5: "Mangue",
    6: "Floresta Alagável (beta)", 49: "Restinga Arborizada", 10: "Formação Natural não Florestal",
    11: "Campo Alagado e Área Pantanosa", 12: "Formação Campestre", 32: "Apicum",
    29: "Afloramento Rochoso", 50: "Restinga Herbácea", 13: "Outras Formações não Florestais",
    14: "Agropecuária", 15: "Pastagem", 18: "Agricultura", 19: "Lavoura Temporária",
    39: "Soja", 20: "Cana", 40: "Arroz", 62: "Algodão (beta)", 41: "Outras Lavouras Temporárias",
    36: "Lavoura Perene", 46: "Café", 47: "Citrus", 48: "Outras Lavouras Perenes",
    9: "Silvicultura", 21: "Mosaico de Usos", 22: "Área não Vegetada", 23: "Praia, Duna e Areal",
    24: "Área Urbanizada", 30: "Mineração", 25: "Outras Áreas não Vegetadas", 26: "Corpo D'água",
    33: "Rio, Lago e Oceano", 31: "Aquicultura", 27: "Não observado"
}

paleta_nomes = {key:value for key, value in zip(dicionario_classes.values(), dicionario_cores.values())}

# Expanded integer palette dictionary preserved as requested for structural integrity
paleta_cores = {
    0: "#ffffff", 1: "#32a65e", 2: "#32a65e", 3: "#1f8d49",
    4: "#7dc975", 5: "#04381d", 6: "#026975", 7: "#000000",
    8: "#000000", 9: "#7a6c00", 10: "#ad975a", 11: "#519799",
    12: "#d6bc74", 13: "#d89f5c", 14: "#FFFFB2", 15: "#edde8e",
    16: "#000000", 17: "#000000", 18: "#f5b3c8", 19: "#C27BA0",
    20: "#db7093", 21: "#ffefc3", 22: "#db4d4f", 23: "#ffa07a",
    24: "#d4271e", 25: "#db4d4f", 26: "#0000FF", 27: "#000000",
    28: "#000000", 29: "#ffaa5f", 30: "#9c0027", 31: "#091077",
    32: "#fc8114", 33: "#2532e4", 34: "#93dfe6", 35: "#9065d0",
    36: "#d082de", 37: "#000000", 38: "#000000", 39: "#f5b3c8",
    40: "#c71585", 41: "#f54ca9", 42: "#cca0d4", 43: "#dbd26b",
    44: "#807a40",45: "#e04cfa", 46: "#d68fe2", 47: "#9932cc",
    48: "#e6ccff", 49: "#02d659", 50: "#ad5100", 51: "#000000",
    52: "#000000", 53: "#000000", 54: "#000000", 55: "#000000",
    56: "#000000", 57: "#CC66FF", 58: "#FF6666", 59: "#006400",
    60: "#8d9e8b", 61: "#f5d5d5", 62: "#ff69b4"
}
palette_list = list(paleta_cores.values())

# Continuous color visualization configuration for Gross Primary Productivity index
vis_gpp = {'min': 0, 'max': 3000, 'palette': ['#f7fcb9', '#addd8e', '#31a354']}



Appending to app.py


 # **4. Load Data & ML Model (Cached)**

In [14]:
%%writefile -a app.py

# ==========================================
# LOAD DATA & ML MODEL (CACHED)
# ==========================================
# Decorator @st.cache_data prevents Streamlit from reloading the CSV on every click
@st.cache_data
def load_csv():
    csv_path = "/content/drive/MyDrive/Colab Notebooks/files/output/final/data_integration_all_areas_final.csv"
    return pd.read_csv(csv_path)

# Decorator @st.cache_resource keeps the ML model in memory for fast inference
@st.cache_resource
def load_ml_model():
    model_path = '/content/drive/MyDrive/Colab Notebooks/files/output/final/modelo_arvore_carbono.pkl'
    try:
        return joblib.load(model_path)
    except Exception as e:
        st.sidebar.error(f"⚠️ Model Load Error: {e}")
        return None

all_farms_df = load_csv()
ml_model = load_ml_model()



Appending to app.py


 # **5. Dashboard Interface & Spatial Processing (Pure Folium)**

In [15]:
%%writefile -a app.py

# ==========================================
# SIDEBAR / USER INTERFACE
# ==========================================
st.sidebar.header("📍 Location Filters")

# Populate state dropdown and filter cities reactively
list_state = sorted(list(all_farms_df['state_name'].unique()))
state_name = st.sidebar.selectbox('State:', list_state)

list_cities = sorted(list(all_farms_df[all_farms_df['state_name'] == state_name]['city_name'].unique()))
city_name = st.sidebar.selectbox('City:', list_cities)

# Filter farms based on selected city
list_farms = sorted(list(all_farms_df[all_farms_df['city_name'] == city_name]['farm'].unique()))
farm_id = st.sidebar.selectbox('Farm (ID):', list_farms)

# ==========================================
# METRICS & AI INFERENCE EXECUTION
# ==========================================
if farm_id:
    # Query specific property data
    data = all_farms_df[all_farms_df['farm'] == farm_id].iloc[0]
    
    st.title(f"🌱 Predicted Carbon Dashboard - Farm {farm_id}")
    st.markdown("---")
    
    # Run the live Artificial Intelligence inference
    ai_prediction_str = "Unavailable"
    if ml_model is not None:
        try:
            # Extract features array matching the exact structure used during training
            features = [[data['area_size']]] 
            prediction_value = ml_model.predict(features)[0]
            ai_prediction_str = f"{prediction_value:.4f}"
        except Exception as ml_err:
            ai_prediction_str = "Inference Error"

    # Render top-level property numerical indicators
    col1, col2, col3, col4, col5 = st.columns(5)
    col1.metric("Area (ha)", f"{data['area_size']:.2f}")
    col2.metric("Bioma", data['biome_name'])
    col3.metric("tCO2/ha (Historical Reality)", f"{data['balance_CO2_ha']:.4f}")
    col4.metric("tCO2/ha (Predicted AI)", ai_prediction_str)
    col5.metric("Total CO2", f"{data['balance_CO2_area']:.2f}")
    
    st.markdown("---")
    
    # ==========================================
    # EARTH ENGINE GEOPROCESSING & FOLIUM MAP
    # ==========================================
    with st.spinner('Loading spatial geometries and satellite imagery...'):
        # Dynamically select regional landlord tenure asset layer
        if state_name == 'Rondônia':
            asset = 'projects/ee-lzfsantos/assets/rg_n_landtenure_imaflora_201810_sirgas'
        elif state_name in ['Minas Gerais', 'Ceará']:
            asset = 'projects/ee-lzfsantos/assets/rg_se_landtenure_imaflora_201810_sirgas'
        elif state_name == 'Rio Grande do Sul':
            asset = 'projects/ee-lzfsantos/assets/rg_s_landtenure_imaflora_201810_sirgas'
        else:
            asset = 'projects/ee-lzfsantos/assets/rg_co_landtenure_imaflora_201810_sirgas'

        # Filter spatial geometry by the chosen farm ID
        farm_feature = ee.FeatureCollection(asset).filter(ee.Filter.eq('id_imovel', int(farm_id)))
        
        map_carbon = mapbiomas.select('gpp_2021')
        clip_carbon = map_carbon.clip(farm_feature)

        # --- NEW RENDERING WITH PURE FOLIUM (NO GEEMAP) ---
        
        # --- NOVA RENDERIZAÇÃO COM FOLIUM PURO (SEM GEEMAP) ---
        
        # 1. Tentar descobrir as coordenadas de forma segura
        try:
            # Pega a primeira fazenda que bate com o ID e extrai as coordenadas
            lon_lat = farm_feature.first().geometry().centroid().coordinates().getInfo()
            center_lon = lon_lat[0]
            center_lat = lon_lat[1]
        except Exception:
            # Escudo: Se o polígono não existir no mapa, avisa na tela e não quebra o site
            st.warning("⚠️ O polígono geométrico desta fazenda não foi encontrado no Earth Engine.")
            center_lat = -15.0 # Centro genérico
            center_lon = -50.0

        # 2. Criar um mapa base usando o Folium padrão
        M = folium.Map(location=[center_lat, center_lon], zoom_start=13, control_scale=True)

        # 2. Create a base map using standard Folium
        M = folium.Map(location=[center_lat, center_lon], zoom_start=13, control_scale=True)

        # 3. Fetch the image URL directly from Google Earth Engine server
        map_id_dict = clip_carbon.getMapId(vis_gpp)
        
        # 4. Add the GEE satellite layer inside the Folium map
        folium.TileLayer(
            tiles=map_id_dict['tile_fetcher'].url_format,
            attr='Google Earth Engine / MapBiomas',
            name='Carbon (GPP)',
            overlay=True,
            control=True
        ).add_to(M)

        # 5. Add layer control (to toggle visualization)
        folium.LayerControl().add_to(M)

        # 6. Display the map on the Streamlit screen using the official Folium bridge
        folium_static(M, width=700, height=500)

        # 7. Native clean HTML/CSS visual legends representation block
        st.markdown("### 📊 Land Use Legend")
        col_leg1, col_leg2, col_leg3 = st.columns(3)
        col_leg1.markdown("🟡 **Pasture** (`#edde8e`)")
        col_leg2.markdown("🟢 **Forest** (`#1f8d49`)")
        col_leg3.markdown("🟣 **Agriculture** (`#E974ED`)")



Appending to app.py


 # **6. Cloud Deployment Execution**

 Run this final cell to handle library updates and start the local tunnel server.

In [ ]:


# 2. Force install latest compatible versions
!pip install -q streamlit==1.29.0 folium streamlit-folium

# 3. Install web tunnel tool globally
!npm install -g localtunnel

# 4. Fetch the Password/IP for authentication
import urllib
print("==========================================================")
print("🔑 SENHA/IP PARA ACESSAR O SITE:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
print("==========================================================")

# 5. Run the dashboard and create the tunnel
!streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false & lt --port 8501

⠙⠹⠸⠼⠴⠦⠧
changed 22 packages in 968ms
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠇==========================================================
🔑 SENHA/IP PARA ACESSAR O SITE: 34.19.73.122
your url is: https://twenty-rats-act.loca.lt



  You can now view your Streamlit app in your browser.

  Network URL: http://172.28.0.12:8501
  External URL: http://34.19.73.122:8501

/content/app.py:215: DeprecationWarning: 
folium_static is deprecated and will be removed in a future release, or
simply replaced with with st_folium which always passes
returned_objects=[] to the component.
Please try using st_folium instead, and
post an issue at https://github.com/randyzwitch/streamlit-folium/issues
if you experience issues with st_folium.

  folium_static(M, width=700, height=500)
/content/app.py:215: DeprecationWarning: 
folium_static is deprecated and will be removed in a future release, or
simply replaced with with st_folium which always passes
returned_objects=[] to the comp

: 